# **3. Style Transfer**

## **3.1. Loss Function**
I used a VGG19 model for style transfer by extracting feature representations from content and style images. I preprocessed the images and computed content loss using MSE at the conv4_2 layer. For style loss, I calculated Gram matrices at five convolutional layers and measured their MSE differences. A truncated VGG19 model served as a feature extractor, and I balanced content and style losses using weights (alpha and beta). This method provides a foundation for optimizing the generated image through iterative updates, effectively blending the style and content characteristics.

In [20]:
# 3.1 Setup — VGG19 feature extractor and image helpers

import os
import zipfile
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision.models import VGG19_Weights
import torchvision.transforms as transforms
from PIL import Image
from tabulate import tabulate

# unzip dataset if not already extracted
ZIP_FILE   = "styles_content.zip"
EXTRACT_TO = "."

if not os.path.isdir("styles") or not os.path.isdir("content"):
    if os.path.exists(ZIP_FILE):
        print(f"Unzipping {ZIP_FILE}...")
        with zipfile.ZipFile(ZIP_FILE, "r") as zf:
            zf.extractall(EXTRACT_TO)
        print("done.")
    else:
        raise FileNotFoundError(f"{ZIP_FILE} not found.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ImageNet mean/std used by VGG
VGG_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
VGG_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)

IMG_SIZE = 256


def load_image(path, size=IMG_SIZE):
    """Open an image, resize it, and convert to tensor in [0,1]."""
    img = Image.open(path).convert("RGB")
    img = img.resize((size, size))
    return img


def to_tensor_no_norm(pil_img):
    """PIL → float32 tensor [0,1] on device, no normalisation yet."""
    t = transforms.ToTensor()(pil_img).unsqueeze(0).to(device)
    return t


def vgg_preprocess(img_tensor):
    """Apply VGG normalisation in-place."""
    return (img_tensor - VGG_MEAN) / VGG_STD


def vgg_unpreprocess(img_tensor):
    """Undo VGG normalisation."""
    return img_tensor * VGG_STD + VGG_MEAN


def clamp_0_1_in_unnormalized_space(img_tensor):
    """Clamp so the unnormalized pixel stays in [0, 1]."""
    with torch.no_grad():
        unnorm = vgg_unpreprocess(img_tensor)
        unnorm.clamp_(0, 1)
        img_tensor.data = vgg_preprocess(unnorm)


# load VGG19 with pretrained weights, freeze everything
vgg = models.vgg19(weights=VGG19_Weights.IMAGENET1K_V1).features.to(device).eval()
for param in vgg.parameters():
    param.requires_grad = False

print("VGG19 loaded and frozen.")

# we'll pull features from these VGG layers
CONTENT_LAYER = "conv4_2"   # one layer for content
STYLE_LAYERS  = ["conv1_1", "conv2_1", "conv3_1", "conv4_1", "conv5_1"]

# map readable names to VGG layer indices
VGG_LAYER_MAP = {
    "conv1_1": 0,  "conv1_2": 2,
    "conv2_1": 5,  "conv2_2": 7,
    "conv3_1": 10, "conv3_2": 12, "conv3_3": 14, "conv3_4": 16,
    "conv4_1": 19, "conv4_2": 21, "conv4_3": 23, "conv4_4": 25,
    "conv5_1": 28, "conv5_2": 30, "conv5_3": 32, "conv5_4": 34,
}


def extract_features(img_tensor, layer_names):
    """Run the image through VGG and return activations at requested layers."""
    feats   = {}
    current = vgg_preprocess(img_tensor)
    for idx, layer in enumerate(vgg):
        current = layer(current)
        for name, target_idx in VGG_LAYER_MAP.items():
            if idx == target_idx and name in layer_names:
                feats[name] = current
    return feats


def gram_matrix(feat):
    """Gram matrix — captures texture/style statistics."""
    b, c, h, w = feat.shape
    f = feat.view(b, c, h * w)
    return torch.bmm(f, f.transpose(1, 2)) / (c * h * w)


def content_loss(gen_feat, content_feat):
    return F.mse_loss(gen_feat, content_feat)


def style_loss(gen_feat, style_feat, weight=1.0):
    return weight * F.mse_loss(gram_matrix(gen_feat), gram_matrix(style_feat))


print("Helper functions defined.")


'content/' already exists, skipping unzip.
'styles/' already exists, skipping unzip.
Using device: cuda

--- Running Style Transfer for 5 Combinations ---
   alpha=1.0, beta=1000.0, steps=50, lr=0.01

> Combo 1:
   Content = bear.jpg
   Style   = Erin-Hanson-Monet's-Bridge.jpg
 [Step 1/50] Content: 0.3993, Style: 3.2781, Total: 3.6774
 [Step 10/50] Content: 0.4996, Style: 1.3320, Total: 1.8317
 [Step 20/50] Content: 0.3717, Style: 1.2718, Total: 1.6435
 [Step 30/50] Content: 0.3288, Style: 1.2075, Total: 1.5363
 [Step 40/50] Content: 0.3164, Style: 1.1254, Total: 1.4418
 [Step 50/50] Content: 0.3134, Style: 1.0336, Total: 1.3470
> Combo 2:
   Content = building.jpg
   Style   = bet-you.jpg
 [Step 1/50] Content: 0.4357, Style: 8.0322, Total: 8.4679
 [Step 10/50] Content: 1.4371, Style: 1.7668, Total: 3.2039
 [Step 20/50] Content: 1.0908, Style: 1.6617, Total: 2.7525
 [Step 30/50] Content: 0.9469, Style: 1.6103, Total: 2.5572
 [Step 40/50] Content: 0.8841, Style: 1.5688, Total: 2.4529
 [

## **3.2. Gradient Descent**
I used a VGG19 model for style transfer, optimizing image pixels using L-BFGS instead of Adam or SGD. The content loss was computed using MSE at the conv4_2 layer, while style loss involved Gram matrices from five convolutional layers. I implemented a closure function to efficiently re-evaluate loss and gradients within L-BFGS steps. To ensure stable optimization, I constrained image updates within the 0-1 range. I experimented with different weight configurations to balance style and content, comparing results between L-BFGS and Adam. Finally, I applied style transfer to a personal photograph

In [11]:
# 3.2 Style Transfer — loss function, optimisers, and experiments
# new imports only (everything else already loaded in 3.1)
import random
import time
import torch.optim as optim
from torchvision.models import vgg19
from torchvision.utils import save_image


def total_loss_func(gen_img, content_img, style_img, c_weight, s_weight):
    """Compute weighted content + style loss for the generated image."""
    all_layers = [CONTENT_LAYER] + STYLE_LAYERS

    c_feats = extract_features(content_img, all_layers)
    s_feats = extract_features(style_img,   all_layers)
    g_feats = extract_features(gen_img,     all_layers)

    # content loss — match activations at conv4_2
    Lc = c_weight * content_loss(g_feats[CONTENT_LAYER], c_feats[CONTENT_LAYER])

    # style loss — match gram matrices across all five style layers
    Ls = 0.0
    for s_lyr in STYLE_LAYERS:
        Ls += style_loss(s_feats[s_lyr], g_feats[s_lyr], weight=s_weight / len(STYLE_LAYERS))

    total = Lc + Ls
    return total, Lc, Ls


def run_lbfgs_until_threshold(
    content_img, style_img,
    c_w, s_w, max_steps=300, threshold=0.5, init_noise_std=0.05
):
    # start from the content image + a little noise, then optimise with L-BFGS
    init_img  = content_img.clone()
    init_img += init_noise_std * torch.randn_like(init_img)
    init_img.clamp_(0, 1)

    gen_img   = nn.Parameter(init_img, requires_grad=True)
    optimizer = optim.LBFGS([gen_img], lr=1.0, max_iter=1)
    records   = []
    step_count = 0

    def closure():
        optimizer.zero_grad()
        total, Lc, Ls = total_loss_func(gen_img, content_img, style_img, c_w, s_w)
        total.backward()
        return total

    while step_count < max_steps:
        closure()
        with torch.no_grad():
            clamp_0_1_in_unnormalized_space(gen_img)

        optimizer.step(closure)
        with torch.no_grad():
            clamp_0_1_in_unnormalized_space(gen_img)

        total_after, Lc_val, Ls_val = total_loss_func(gen_img, content_img, style_img, c_w, s_w)
        step_count += 1
        records.append((step_count, total_after.item(), Lc_val.item(), Ls_val.item()))

        if total_after.item() < threshold:
            print(f"Early stopping at step {step_count}, total_loss={total_after.item():.4f}")
            break

    return gen_img.detach(), records


def run_adam_until_threshold(
    content_img, style_img,
    c_w, s_w, max_steps=500, threshold=0.5,
    lr=0.001, init_noise_std=0.02
):
    # same idea as L-BFGS but using Adam — usually needs more steps
    init_img  = content_img.clone()
    init_img += init_noise_std * torch.randn_like(init_img)
    init_img.clamp_(0, 1)

    gen_img   = nn.Parameter(init_img, requires_grad=True)
    optimizer = optim.Adam([gen_img], lr=lr)
    records   = []

    for step in range(max_steps):
        optimizer.zero_grad()
        total, Lc, Ls = total_loss_func(gen_img, content_img, style_img, c_w, s_w)
        total.backward()
        with torch.no_grad():
            clamp_0_1_in_unnormalized_space(gen_img)
        optimizer.step()

        total_val, Lc_val, Ls_val = total_loss_func(gen_img, content_img, style_img, c_w, s_w)
        records.append((step + 1, total_val.item(), Lc_val.item(), Ls_val.item()))

        if total_val.item() < threshold:
            print(f"Early stopping at step {step+1}, total_loss={total_val.item():.4f}")
            break

    with torch.no_grad():
        clamp_0_1_in_unnormalized_space(gen_img)
    return gen_img.detach(), records


def vgg2png(img_tensor, filename="output.png"):
    """Save a VGG-normalised tensor as a PNG."""
    with torch.no_grad():
        clamp_0_1_in_unnormalized_space(img_tensor)
    unnorm = vgg_unpreprocess(img_tensor)
    unnorm.clamp_(0, 1)
    save_image(unnorm, filename)
    print(f"\033[1;32mSaved:\033[0m {filename}")


def style_experiment_5pairs(
    image_pairs,
    c_weights=[1.0, 1.0, 2.0, 5.0, 10.0],
    s_weights=[1.0, 5.0, 5.0, 2.0,  1.0],
    threshold=0.5,
    lbfgs_steps=300,
    adam_steps=500
):
    # run 5 content/style weight combos on each image pair,
    # using both L-BFGS and Adam, stopping early if loss drops below threshold
    all_results = []

    for pair_idx, (cpath, spath) in enumerate(image_pairs, start=1):
        print(f"\n\033[1;36m--- Pair #{pair_idx}: content='{cpath}', style='{spath}' ---\033[0m")

        cimg_pil = load_image(os.path.join("content", cpath))
        simg_pil = load_image(os.path.join("styles",  spath))
        cimg     = to_tensor_no_norm(cimg_pil)
        simg     = to_tensor_no_norm(simg_pil)

        pair_results = []
        for i in range(len(c_weights)):
            cw = c_weights[i]
            sw = s_weights[i]
            print(f"\nConfig {i+1}: (cw={cw}, sw={sw})")

            # L-BFGS run
            start_lbfgs  = time.time()
            final_lbfgs, rec_lbfgs = run_lbfgs_until_threshold(
                cimg, simg, cw, sw, max_steps=lbfgs_steps, threshold=threshold
            )
            end_lbfgs = time.time()

            # Adam run
            start_adam  = time.time()
            final_adam, rec_adam = run_adam_until_threshold(
                cimg, simg, cw, sw, max_steps=adam_steps, threshold=threshold, lr=0.001
            )
            end_adam = time.time()

            lbfgs_total = rec_lbfgs[-1][1]
            lbfgs_cLoss = rec_lbfgs[-1][2]
            lbfgs_sLoss = rec_lbfgs[-1][3]

            adam_total  = rec_adam[-1][1]
            adam_cLoss  = rec_adam[-1][2]
            adam_sLoss  = rec_adam[-1][3]

            lbfgs_filename = f"pair{pair_idx}_conf{i+1}_LBFGS.png"
            adam_filename  = f"pair{pair_idx}_conf{i+1}_ADAM.png"
            vgg2png(final_lbfgs, lbfgs_filename)
            vgg2png(final_adam,  adam_filename)

            pair_results.append([
                f"Pair{pair_idx}-Conf{i+1}",
                f"(cw={cw},sw={sw})",
                f"{lbfgs_total:.5f}", f"{lbfgs_cLoss:.5f}", f"{lbfgs_sLoss:.5f}", f"{(end_lbfgs - start_lbfgs):.2f}s",
                f"{adam_total:.5f}",  f"{adam_cLoss:.5f}",  f"{adam_sLoss:.5f}",  f"{(end_adam  - start_adam):.2f}s"
            ])

        all_results.extend(pair_results)

    return all_results


if __name__ == "__main__":

    image_pairs = [
        ("bear.jpg",    "bet-you.jpg"),
        ("cat.jpg",     "horse-cart.jpg"),
        ("picnic.jpg",  "the-couple.jpg"),
        ("town.jpg",    "zebra.jpg"),
        ("building.jpg","not-detected.jpg"),
    ]

    final_table = style_experiment_5pairs(
        image_pairs=image_pairs,
        c_weights=[1.0, 1.0, 2.0, 5.0, 10.0],
        s_weights=[1.0, 5.0, 5.0, 2.0,  1.0],
        threshold=0.5,
        lbfgs_steps=300,
        adam_steps=500
    )

    headers = [
        "Pair-Config", "(c_w, s_w)",
        "LBFGS:Total", "LBFGS:Cont", "LBFGS:Styl", "LBFGS Time",
        "ADAM:Total",  "ADAM:Cont",  "ADAM:Styl",  "ADAM Time"
    ]
    print("\n\033[1;36m=== Final Style Transfer Results for 5 Pairs x 5 Configurations ===\033[0m")
    print(tabulate(final_table, headers=headers, tablefmt="fancy_grid"))
    print("\n\033[1;32mDone! Check the saved PNG files for each pair & config.\033[0m")

    # apply style to your own photo using the best config
    print("\033[1;35mAlso transferring style to your personal photo...\033[0m")

    my_photo_path = os.path.join("content", "my_photo.jpg")
    if not os.path.isfile(my_photo_path):
        print(f"\033[1;31mERROR:\033[0m Cannot find '{my_photo_path}'. Place your photo there.")
    else:
        my_photo_img_pil = load_image(my_photo_path)
        my_photo_img     = to_tensor_no_norm(my_photo_img_pil)

        my_style_path = os.path.join("styles", "the-couple.jpg")
        if not os.path.isfile(my_style_path):
            print(f"\033[1;31mERROR:\033[0m Cannot find '{my_style_path}'.")
        else:
            my_style_img_pil = load_image(my_style_path)
            my_style_img     = to_tensor_no_norm(my_style_img_pil)

            final_myphoto, rec_my = run_lbfgs_until_threshold(
                my_photo_img, my_style_img,
                c_w=1.0, s_w=5.0,
                max_steps=200, threshold=0.5
            )
            vgg2png(final_myphoto, "myphoto_styled_LBFGS.png")

    print("\n\033[1;32mAll done!\033[0m")


'content' folder already exists, skipping unzip.

'styles' folder already exists, skipping unzip.

Content images: ['cat.jpg', 'picnic.jpg', 'cows.jpg', 'my_photo.jpg', 'cat-on-table.jpg', 'building.jpg', 'bear.jpg', 'white-building.jpg', 'mountains.jpg', 'town.jpg', 'cat-sleeping.jpg']
Style images:   ['bet-you.jpg', 'not-detected.jpg', 'head-of-paula-eyles.jpg', 'landscape-with-a-palace.jpg', 'zebra.jpg', 'the-couple.jpg', 'the-exit-of-the-russian-ballet.jpg', "Erin-Hanson-Monet's-Bridge.jpg", 'horse-cart.jpg', 'red-sea-passage.jpg']

Using device: cuda

Loading VGG19 (IMAGENET1K_V1) ...
VGG19 loaded successfully.


--- Pair #1: content='bear.jpg', style='bet-you.jpg' ---

Config 1: (cw=1.0, sw=1.0)
Early stopping at step 8, total_loss=0.4745
Early stopping at step 2, total_loss=0.4377
Saved final image: pair1_conf1_LBFGS.png
Saved final image: pair1_conf1_ADAM.png

Config 2: (cw=1.0, sw=5.0)
Early stopping at step 8, total_loss=0.4822
Early stopping at step 2, total_loss=0.4358
Save